In [0]:
dbutils.widgets.text("bronze_layer", "bronze")
bronze = dbutils.widgets.get("bronze_layer")

dbutils.widgets.text("silver_layer", "silver")
silver = dbutils.widgets.get("silver_layer")

dbutils.widgets.text("schema_name", "fhir")
schema = dbutils.widgets.get("schema_name")

dbutils.widgets.text("tables", "Patient,Encounter,Observation,Condition")
tables_str = dbutils.widgets.get("tables")

Tables = [t.strip() for t in tables_str.split(",")]

In [0]:
table = None
for i in Tables:
    if i.lower() == "patient":
        table = i
        break

if table is None:
    dbutils.notebook.exit("Exiting notebook: Table is not condition")

print(table + " table is present ")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

### Assigning the value to the variable and loading the data into df based on if the bronze table exist or not.

In [0]:
bronze_table = f"{bronze}.{schema}.{table}"
silver_table = f"{silver}.{schema}.{table}"

if spark.catalog.tableExists(silver_table):
    bronze_df = spark.table(bronze_table).filter(
        col("load_date") >= date_sub(current_date(), 4)
    )
else:
    bronze_df = spark.table(bronze_table)

bronze_df.select("load_date").show()

### Exploding and flattening the raw_JSON from df created from bronze table.


In [0]:
parsed_df = bronze_df.withColumn(
    "json_data",
    from_json(
        col("raw_json"),
        "array<struct<entry:array<struct<resource:struct<id:string,active:boolean,gender:string,birthDate:string,name:array<struct<family:string,given:array<string>>>,telecom:array<struct<system:string,value:string>>,address:array<struct<city:string,state:string,postalCode:string,country:string>>,maritalStatus:struct<coding:array<struct<code:string,display:string>>>,meta:struct<versionId:string,lastUpdated:string>>>>>>"
    )
)

In [0]:
exploded_df = parsed_df.select(
    explode(col("json_data")).alias("bundle"),
    "extraction_timestamp"
)

entry_df = exploded_df.select(
    explode(col("bundle.entry")).alias("entry"),
    "extraction_timestamp"
)

In [0]:
staging_df = entry_df.select(
    col("entry.resource.id").alias("patient_id"),
    col("entry.resource.active").alias("active"),
    col("entry.resource.gender").alias("gender"),
    col("entry.resource.birthDate").alias("birth_date"),

    # Name
    get(col("entry.resource.name"), 0)["family"].alias("family_name"),
    get(get(col("entry.resource.name"), 0)["given"], 0).alias("given_name"),

    # Telecom (SAFE)
    get(col("entry.resource.telecom"), 0)["value"].alias("contact_1"),
    get(col("entry.resource.telecom"), 1)["value"].alias("contact_2"),

    # Address (SAFE)
    get(col("entry.resource.address"), 0)["city"].alias("city"),
    get(col("entry.resource.address"), 0)["state"].alias("state"),
    get(col("entry.resource.address"), 0)["postalCode"].alias("postal_code"),
    get(col("entry.resource.address"), 0)["country"].alias("country"),

    # Marital
    get(col("entry.resource.maritalStatus.coding"), 0)["code"].alias("marital_code"),
    get(col("entry.resource.maritalStatus.coding"), 0)["display"].alias("marital_status"),

    col("entry.resource.meta.versionId").alias("version_id"),
    col("entry.resource.meta.lastUpdated").alias("last_updated"),

)

In [0]:
staging_df = staging_df \
    .withColumn("effective_start_date", col("last_updated").cast("timestamp")) \
    .withColumn("effective_end_date", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True))

#staging_df.explain(True)
staging_df.show(2)

### Removing duplicate PK records based on last_updated date. So, only records having latest last_updated date will be considered.

In [0]:
window = Window.partitionBy("patient_id").orderBy(col("last_updated").desc())

staging_df = staging_df \
    .withColumn("rn", row_number().over(window)) \
    .filter("rn = 1") \
    .drop("rn")


### Checking if silver table exist or not if it exist than the the code will go to merge in case if silver table now exist df will directly created the table and exit the notebook

In [0]:
silver_table = f"{silver}.{schema}.{table}"
print(silver_table)

if not spark.catalog.tableExists(silver_table):
    staging_df.write.format("delta").saveAsTable(silver_table)
    dbutils.notebook.exit("Exiting notebook: As this was the first time load")

In [0]:
staging_df.createOrReplaceTempView("staging_patient")

### SCD 2 Type loading data into silver table. Therefore, 3 things are handle - 
1. if an update is found for a record it will inactive the older record. 
2. if a new record is found it will insert it. 
3. and than it will insert the update record.

In [0]:
query = f"""
MERGE INTO {silver_table} AS tgt
USING staging_patient AS src
ON tgt.patient_id = src.patient_id
AND tgt.is_current = true

WHEN MATCHED AND (
    tgt.active <> src.active OR
    tgt.gender <> src.gender OR
    tgt.birth_date <> src.birth_date OR
    tgt.family_name <> src.family_name OR
    tgt.given_name <> src.given_name OR
    tgt.city <> src.city OR
    tgt.state <> src.state OR
    tgt.country <> src.country OR
    tgt.marital_code <> src.marital_code
)
THEN UPDATE SET
    tgt.is_current = false,
    tgt.effective_end_date = current_timestamp()

WHEN NOT MATCHED
THEN INSERT (
    patient_id,
    active,
    gender,
    birth_date,
    family_name,
    given_name,
    city,
    state,
    country,
    marital_code,
    last_updated,
    effective_start_date,
    effective_end_date,
    is_current
)
VALUES (
    src.patient_id,
    src.active,
    src.gender,
    src.birth_date,
    src.family_name,
    src.given_name,
    src.city,
    src.state,
    src.country,
    src.marital_code,
    src.last_updated,
    current_timestamp(),
    NULL,
    true
)
"""

query1 = f"""
INSERT INTO {silver_table} (
    patient_id,
    active,
    gender,
    birth_date,
    family_name,
    given_name,
    city,
    state,
    country,
    marital_code,
    last_updated,
    effective_start_date,
    effective_end_date,
    is_current
)
SELECT 
    src.patient_id,
    src.active,
    src.gender,
    src.birth_date,
    src.family_name,
    src.given_name,
    src.city,
    src.state,
    src.country,
    src.marital_code,
    src.last_updated,
    current_timestamp() AS effective_start_date,
    NULL AS effective_end_date,
    true AS is_current
FROM staging_patient src
JOIN {silver_table} tgt
    ON tgt.patient_id = src.patient_id
WHERE 
    tgt.is_current = false
AND (
    tgt.active <> src.active OR
    tgt.gender <> src.gender OR
    tgt.birth_date <> src.birth_date OR
    tgt.family_name <> src.family_name OR
    tgt.given_name <> src.given_name OR
    tgt.city <> src.city OR
    tgt.state <> src.state OR
    tgt.country <> src.country OR
    tgt.marital_code <> src.marital_code
)
AND NOT EXISTS (
    SELECT 1
    FROM {silver_table} t2
    WHERE t2.patient_id = src.patient_id
      AND t2.is_current = true
)
"""

spark.sql(query)
spark.sql(query1)